# 01 · 데이터 크롤링 워크플로

이 노트북은 서울시 70개 미술관 정문 이미지를 수집하고 전처리하는 과정을 담고 있습니다.

## 단계
- 작업 환경 준비
- 미술관 메타데이터 로드
- 이미지 크롤러 실행
- 품질 필터링 및 중복 제거
- Google Drive / 외부 스토리지 업로드 옵션



In [ ]:
# 환경 설정 (Colab에서 실행 시 주석 해제)
# !pip install -q pillow tqdm requests icrawler pandas

import os
import json
from pathlib import Path
from typing import List

import pandas as pd
from icrawler.builtin import BingImageCrawler



In [ ]:
# 미술관 목록 정의 (예시)
museum_list = [
    {"name": "국립현대미술관 서울관", "query": "국립현대미술관 서울관 정문"},
    {"name": "서울시립미술관", "query": "서울시립미술관 본관 정문"},
    # TODO: 실제 70개 목록을 CSV로 준비해 읽어오세요.
]

museum_df = pd.DataFrame(museum_list)
museum_df.head()


In [ ]:
OUTPUT_DIR = Path('../data/seoul_museums')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def crawl_images(name: str, query: str, max_num: int = 40):
    target_dir = OUTPUT_DIR / name
    target_dir.mkdir(parents=True, exist_ok=True)
    crawler = BingImageCrawler(storage={'root_dir': str(target_dir)})
    crawler.crawl(keyword=query, max_num=max_num, file_idx_offset='auto')
    print(f"[DONE] {name} ({max_num}장) → {target_dir}")

# 예시 실행
for row in museum_df.itertuples():
    crawl_images(row.name, row.query, max_num=5)



In [ ]:
from PIL import Image
import imagehash

def filter_duplicates(folder: Path, hash_func=imagehash.phash, threshold: int = 5):
    hashes = {}
    removed = 0
    for image_path in folder.glob('*.jpg'):
        try:
            with Image.open(image_path) as img:
                img_hash = hash_func(img)
        except Exception as exc:
            print(f"[WARN] {image_path.name} 열기 실패: {exc}")
            image_path.unlink(missing_ok=True)
            removed += 1
            continue
        for stored_hash, stored_path in hashes.items():
            if abs(img_hash - stored_hash) <= threshold:
                image_path.unlink(missing_ok=True)
                removed += 1
                break
        else:
            hashes[img_hash] = image_path
    return removed

removed_total = 0
for subdir in OUTPUT_DIR.iterdir():
    if subdir.is_dir():
        removed_total += filter_duplicates(subdir)
print(f"중복/오류 이미지 {removed_total}장 제거")


## 외부 스토리지 업로드 예시

```python
from google.colab import drive
from pathlib import Path

# Colab 전용
# drive.mount('/content/drive')

backup_dir = Path('/content/drive/MyDrive/seoul-museum-ai/data/seoul_museums')
backup_dir.mkdir(parents=True, exist_ok=True)

!rsync -av --progress ../data/seoul_museums/ "$backup_dir/"
```

필요에 따라 AWS S3, Oracle Object Storage, Wasabi 등의 CLI를 활용해 백업 스크립트를 작성하세요.

